In [1]:
import os
from dotenv import load_dotenv
load_dotenv() 


True

In [2]:
PAGE_ID=os.getenv("PAGE_ID")
ACCESS_TOKEN='EAFrqKElW2awBQYutGMV1FoldQg1p28sjbC0jGI4mQWqtUsZCb6cmu5uydH7R9otUe0EeDadMnwvXHEu5QZCA0NNPIUZBd1hFUVjRVdyuAtw3mZANVQXIe3mMJZAQl3INXevqqqg5KbMRF33SQxanZCUIXQGryip8iqjCoDnFLm51pZAs6cweXkeBekT3ZBsW01fVhD38'

In [3]:
import requests

PUBLIC_VIDEO_URL = 'https://www.muscleandstrength.com/video/highinvertedrow.mp4' # Must be a public URL
VIDEO_TITLE = 'My Demo Video'
VIDEO_DESCRIPTION = 'This video is example upload from mp4 url.'


In [14]:

url = f"https://graph-video.facebook.com/v18.0/{PAGE_ID}/videos"
payload = {
    'title': VIDEO_TITLE,
    'description': VIDEO_DESCRIPTION,
    'file_url': PUBLIC_VIDEO_URL,
    'access_token': ACCESS_TOKEN
}

response = requests.post(url, data=payload)


In [15]:

if response.status_code == 200:
    result = response.json()
    print(f"Video upload initiated successfully. Video ID: {result.get('id')}")
else:
    print(f"Error uploading video by URL: {response.text}")



Video upload initiated successfully. Video ID: 3180767115437975


### Facebook vidoe dircet uplaod multi step process

In [10]:
import requests
import os
import json

# Configuration
VIDEO_FILE_PATH = "ForBiggerEscapes.mp4"
VIDEO_TITLE = 'My ForBiggerEscapes Video'
VIDEO_DESCRIPTION = 'ForBiggerEscapes!'

# 1. Start the upload session
video_len = os.path.getsize(VIDEO_FILE_PATH)
start_url = f"https://graph-video.facebook.com/v18.0/{PAGE_ID}/videos" # Use the latest API version
params_start = {
    'upload_phase': 'start',
    'file_size': video_len,
    'access_token': ACCESS_TOKEN
}
# VIDEO_FILE_PATH


In [11]:

response_start = requests.post(start_url, params=params_start)
if response_start.status_code != 200:
    print(f"Error starting upload: {response_start.text}")
    exit()



In [12]:

first_step_json = response_start.json()
upload_session_id = first_step_json['upload_session_id']
# You can also get a specific upload_url from the response in some cases

print(f"Upload session started with ID: {upload_session_id}")


Upload session started with ID: 860493196880322


In [2]:
first_step_json

NameError: name 'first_step_json' is not defined

In [ ]:

# 2. Transfer the video file (simple, single chunk upload for small files)
# For large files, you would need to split into chunks and loop
with open(VIDEO_FILE_PATH, 'rb') as video_file:
    files = {'video_file_chunk': open(VIDEO_FILE_PATH, 'rb')}
    params_transfer = {
        'upload_phase': 'transfer',
        'upload_session_id': upload_session_id,
        'start_offset': 0, # The offset should be 0 for the first/only chunk
        'access_token': ACCESS_TOKEN
    }
    
    # Note: Using 'files' parameter in requests automatically uses multipart/form-data
    response_transfer = requests.post(
        "https://rupload.facebook.com", # Use rupload host for transfer
        params=params_transfer,
        files=files
    )

if response_transfer.status_code != 200:
    print(f"Error during transfer: {response_transfer.text}")
    exit()

print("Video file transferred successfully.")
files['video_file_chunk'].close() # Close the file explicitly


Error during transfer: {"debug_info":{"retriable":false,"type":"InvalidEndpointError","message":"Endpoint / doesn't exist"}}
Video file transferred successfully.


: 

In [ ]:

# 3. Finish the upload and publish
params_finish = {
    'upload_phase': 'finish',
    'upload_session_id': upload_session_id,
    'access_token': ACCESS_TOKEN,
    'title': VIDEO_TITLE,
    'description': VIDEO_DESCRIPTION
}

response_finish = requests.post(start_url, params=params_finish)

if response_finish.status_code == 200:
    result = response_finish.json()
    print(f"Video published successfully. Post ID: {result.get('post_id')}, Video ID: {result.get('id')}")
else:
    print(f"Error finishing upload: {response_finish.text}")